# pix2pix — Image-to-Image Translation with Conditional Adversarial Networks (2016)

_Papers · Generative Models_

**Turn one image into another (edges&rarr;photo, labels&rarr;street-scene) with a U-Net generator judged by a patch-wise GAN, plus an L1 term — one recipe for dozens of translation tasks.**

---

This notebook is a practice scaffold for the **pix2pix — Image-to-Image Translation with Conditional Adversarial Networks (2016)** lesson. We build the recipe one piece at a time — the paired toy data, the U-Net generator, the PatchGAN discriminator, and the combined loss — running and reading each step before moving on. Then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Sanity-check the loss arithmetic and build the paired data

First we recompute the lesson's worked numbers so the pieces of pix2pix's Eq. 4 objective are concrete: the L1 distance between two pixels, the `lambda=100` weighting, and the two cGAN log terms. Then we build a tiny **paired** dataset — an outline (the input) mapped to a disk filled with stripes (the target). The stripe *phase* is chosen at random, so several valid targets share one input: that multimodality is exactly the regime where L1 alone blurs.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math

torch.manual_seed(0)
np.random.seed(0)

# --- 0. Sanity-check the worked example (loss arithmetic). ---
l1_one_pixel = abs(0.7 - (-0.3))            # |0.7 - (-0.3)| = 1.0
lambda_term = 100 * l1_one_pixel            # lambda * L1 = 100.0
cgan_real_term = math.log(0.9)              # log D when D=0.9  -> -0.1054
cgan_fake_term = math.log(1 - 0.2)          # log(1-D) when D=0.2 -> -0.2231
patch_equilibrium = -math.log(0.5)          # -log 0.5 -> 0.6931

print("worked example:")
print("  L1 of one pixel |0.7-(-0.3)| = %.4f" % l1_one_pixel)
print("  lambda * that = 100 * 1.0    = %.1f" % lambda_term)
print("  cGAN real term  log D (D=0.9)= %.4f" % cgan_real_term)
print("  cGAN fake term  log(1-D) D=.2= %.4f" % cgan_fake_term)
print("  patch equilibrium -log 0.5   = %.4f" % patch_equilibrium)

# --- 1. Toy PAIRED data: outline (input) -> disk filled with stripes (target). ---
# The stripe PHASE is random => several valid targets per input (multimodal).
H = 16
CY = CX = 8
R = 5
yy, xx = np.mgrid[0:H, 0:H]
DIST = np.sqrt((yy - CY)**2 + (xx - CX)**2)
INSIDE = DIST <= R
OUTLINE = (INSIDE & (DIST > R - 1.3)).astype(np.float32)

def make_pair():
    phase = np.random.randint(0, 2)                      # <- the multimodality
    stripes = ((xx + phase) % 2 == 0).astype(np.float32)
    filled = np.where(INSIDE, stripes, 0.0).astype(np.float32)
    return OUTLINE.copy(), filled

def batch(m):
    xs, ys = zip(*[make_pair() for _ in range(m)])
    x = torch.tensor(np.stack(xs)).view(m, 1, H, H)
    y = torch.tensor(np.stack(ys)).view(m, 1, H, H)
    return x * 2 - 1, y * 2 - 1                           # to [-1,1] for tanh

### Step 2 — Define the U-Net generator and the PatchGAN discriminator

The generator is a small **U-Net**: an encoder downsamples to a bottleneck, a decoder upsamples back, and a **skip connection** carries the encoder feature `e1` straight onto the matching decoder layer so fine spatial detail is not lost. The `Dropout(0.5)` in the decoder acts as the generator's only source of noise. The discriminator is a **PatchGAN**: it takes the *pair* `(input, output)` concatenated on the channel axis and emits a grid of logits — one realness verdict per local patch, not a single scalar.

In [ ]:
# --- 2. U-Net generator: encoder/decoder + ONE skip connection + dropout-as-noise. ---
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = nn.Sequential(nn.Conv2d(1, 16, 4, 2, 1), nn.LeakyReLU(0.2))           # 16->8
        self.d2 = nn.Sequential(nn.Conv2d(16, 32, 4, 2, 1), nn.LeakyReLU(0.2))          # 8->4
        self.u1 = nn.Sequential(nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.ReLU(),
                                nn.Dropout(0.5))                                        # 4->8 ; dropout = noise
        self.u2 = nn.Sequential(nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh())          # 8->16 ; in=32 after skip

    def forward(self, x):
        e1 = self.d1(x)
        e2 = self.d2(e1)
        d = self.u1(e2)
        d = torch.cat([d, e1], 1)                         # <- U-Net SKIP: encoder e1 onto decoder
        return self.u2(d)

# --- 3. PatchGAN discriminator: input is the PAIR (x, output); output is a GRID of logits. ---
class PatchD(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 16, 4, 2, 1), nn.LeakyReLU(0.2),  # in=2: concat(x, output)
            nn.Conv2d(16, 32, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 1, 3, 1, 1))                     # 4x4 grid of patch logits (NOT one scalar)

    def forward(self, x, y):
        pair = torch.cat([x, y], 1)
        return self.net(pair)

### Step 3 — Train the generator (with and without the GAN term)

Here is the training loop for pix2pix's Eq. 4 objective. Each step the discriminator learns to call the real pair "real" and the generated pair "fake" *per patch*; then the generator takes a step that (a) fools the discriminator and (b) stays close to the target via the L1 term weighted by `lambda=100`. We run it two ways: `use_gan=False` is the **ablation** (L1 only), and `use_gan=True` is the full objective — so we can compare what each term buys.

In [ ]:
bce = nn.BCEWithLogitsLoss()
L1 = nn.L1Loss()
LAM = 100.0                                              # lambda=100 (Eq. 4)

def train(use_gan, steps=2500):
    torch.manual_seed(1)
    G = UNet()
    optG = torch.optim.Adam(G.parameters(), lr=2e-3, betas=(0.5, 0.999))
    if use_gan:
        D = PatchD()
        optD = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.5, 0.999))
    for s in range(steps):
        x, y = batch(64)
        # (a) DISCRIMINATOR step: real pair "real", generated pair "fake" (per patch).
        if use_gan:
            fake = G(x).detach()                          # detach: this step must NOT move G
            dr = D(x, y)
            df = D(x, fake)
            lossD = bce(dr, torch.ones_like(dr)) + bce(df, torch.zeros_like(df))
            optD.zero_grad()
            lossD.backward()
            optD.step()
        # (b) GENERATOR step: fool D (if used) AND stay near the target -> Eq. 4.
        fake = G(x)
        lossL1 = L1(fake, y)
        if use_gan:
            dg = D(x, fake)
            lossG = bce(dg, torch.ones_like(dg)) + LAM * lossL1     # cGAN + lambda*L1
        else:
            lossG = LAM * lossL1                                    # ABLATION: L1 only
        optG.zero_grad()
        lossG.backward()
        optG.step()
        if s % 500 == 0:
            tag = "L1+GAN" if use_gan else "L1only"
            print("  [%s] step %4d  L1 %.4f" % (tag, s, lossL1.item()))
    return G

### Step 4 — Compare the two generators side by side

Finally we render each generator's output as ASCII art and measure the **interior sharpness** — the spatial standard deviation of pixels inside the disk. A value near `0` means a flat gray blur (the model hedged across stripe phases); a value near `1` means it committed to one sharp phase. This is the paper's central ablation reproduced on toy data: the GAN term buys sharpness, the L1 term buys faithfulness to the shape, and Eq. 4 wants both.

In [ ]:
def show(G, title):
    torch.manual_seed(7)
    x, y = batch(1)
    with torch.no_grad():
        out = G(x)[0, 0]
    print(title)
    palette = " .:-=+*#%@"
    for r in range(H):
        row_chars = []
        for c in range(H):
            brightness = ((out[r, c] + 1) / 2).clamp(0, 1)   # map [-1,1] -> [0,1]
            level = min(9, int(brightness * 9))
            row_chars.append(palette[level])
        print("".join(row_chars))

def sharp_std(G, n=2000):           # spatial STD inside the disk: 0=flat gray blur, ~1=committed stripes
    x, y = batch(n)
    with torch.no_grad():
        out = G(x)
    mask = torch.tensor(INSIDE).view(1, 1, H, H).expand(n, 1, H, H).bool()
    vals = out[mask].view(n, -1)
    return vals.std(dim=1).mean().item()

print("\n--- train L1 ONLY (ablation) ---")
Gl1 = train(use_gan=False)
print("\n--- train L1 + PatchGAN (Eq. 4) ---")
Ggan = train(use_gan=True)

show(Gl1,  "\nL1-only output  (expect washed-out / gray = blur):")
show(Ggan, "\nL1+GAN output   (expect a committed sharp stripe fill):")

print("\ninterior sharpness = spatial std inside the disk (0=gray blur, ~1=committed stripes):")
print("  L1 only  : %.3f   <- flat gray: averaged the two stripe phases (the blur)" % sharp_std(Gl1))
print("  L1 + GAN : %.3f   <- committed to one sharp phase (matches the target)"   % sharp_std(Ggan))
print("The GAN term buys SHARPNESS; the L1 term buys FAITHFULNESS (shape). Eq. 4 wants both.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_On a paired task with an ambiguous target, what does each loss term buy — does L1 alone blur, and does adding the PatchGAN make the output sharp?_

### Step 1 — Rebuild the task and the two networks

This visualization cell is self-contained, so it re-declares the paired stripe-fill task and the U-Net / PatchGAN models exactly as above. We reset the seeds first so the run is reproducible. Nothing here is new — it is the same multimodal regime (pix2pix Fig. 4) where L1 blurs and the PatchGAN sharpens.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# Toy PAIRED task: outline (input) -> disk filled with stripes (target), stripe PHASE random
# => multimodal target. This is the regime where L1 blurs and the PatchGAN sharpens (pix2pix Fig.4).
H = 16
CY = CX = 8
R = 5
yy, xx = np.mgrid[0:H, 0:H]
DIST = np.sqrt((yy - CY)**2 + (xx - CX)**2)
INSIDE = DIST <= R
OUTLINE = (INSIDE & (DIST > R - 1.3)).astype(np.float32)

def make_pair():
    phase = np.random.randint(0, 2)
    stripes = ((xx + phase) % 2 == 0).astype(np.float32)
    filled = np.where(INSIDE, stripes, 0.0).astype(np.float32)
    return OUTLINE.copy(), filled

def batch(m):
    xs, ys = zip(*[make_pair() for _ in range(m)])
    x = torch.tensor(np.stack(xs)).view(m, 1, H, H) * 2 - 1
    y = torch.tensor(np.stack(ys)).view(m, 1, H, H) * 2 - 1
    return x, y

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = nn.Sequential(nn.Conv2d(1, 16, 4, 2, 1), nn.LeakyReLU(0.2))
        self.d2 = nn.Sequential(nn.Conv2d(16, 32, 4, 2, 1), nn.LeakyReLU(0.2))
        self.u1 = nn.Sequential(nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.ReLU(), nn.Dropout(0.5))
        self.u2 = nn.Sequential(nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh())

    def forward(self, x):
        e1 = self.d1(x)
        e2 = self.d2(e1)
        d = self.u1(e2)
        d = torch.cat([d, e1], 1)
        return self.u2(d)

class PatchD(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(2, 16, 4, 2, 1), nn.LeakyReLU(0.2),
                                 nn.Conv2d(16, 32, 4, 2, 1), nn.LeakyReLU(0.2),
                                 nn.Conv2d(32, 1, 3, 1, 1))

    def forward(self, x, y):
        return self.net(torch.cat([x, y], 1))

### Step 2 — Train both variants and log the L1 history

Now we train the L1-only and L1+GAN generators with identical settings, recording the L1 distance every 100 steps so we can see each curve's trajectory. Then we measure the interior sharpness of each. The printout makes the headline numbers explicit: L1 alone hedges the ambiguous fill toward flat gray (std near 0), while L1+PatchGAN commits to one sharp phase (std near 1).

In [ ]:
bce = nn.BCEWithLogitsLoss()
L1 = nn.L1Loss()
LAM = 100.0

def train(use_gan, steps=2500):
    torch.manual_seed(1)
    G = UNet()
    optG = torch.optim.Adam(G.parameters(), lr=2e-3, betas=(0.5, 0.999))
    if use_gan:
        D = PatchD()
        optD = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.5, 0.999))
    hist = []
    for s in range(steps):
        x, y = batch(64)
        if use_gan:
            fake = G(x).detach()
            dr = D(x, y)
            df = D(x, fake)
            lossD = bce(dr, torch.ones_like(dr)) + bce(df, torch.zeros_like(df))
            optD.zero_grad()
            lossD.backward()
            optD.step()
        fake = G(x)
        lossL1 = L1(fake, y)
        if use_gan:
            dg = D(x, fake)
            lossG = bce(dg, torch.ones_like(dg)) + LAM * lossL1
        else:
            lossG = LAM * lossL1
        optG.zero_grad()
        lossG.backward()
        optG.step()
        if s % 100 == 0:
            hist.append((s, round(lossL1.item(), 4)))
    return G, hist

def sharp_std(G, n=2000):              # spatial std inside the disk: 0=flat gray blur, ~1=committed stripes
    x, y = batch(n)
    with torch.no_grad():
        out = G(x)
    mask = torch.tensor(INSIDE).view(1, 1, H, H).expand(n, 1, H, H).bool()
    return out[mask].view(n, -1).std(dim=1).mean().item()

Gl1, hl1 = train(False)
Ggan, hgan = train(True)

print("interior sharpness std (0=gray blur, ~1=committed): L1only %.3f  L1+GAN %.3f"
      % (sharp_std(Gl1), sharp_std(Ggan)))
print("L1-only L1 history :", hl1)
print("L1+GAN  L1 history :", hgan)
# L1 alone hedges the ambiguous fill to a flat gray (std ~0 = blur); L1+PatchGAN commits to one sharp phase (std ~1).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The loss ablation (the heart of the paper). Take your working pix2pix on the ambiguous-fill task
            and train it three ways: L1 only, GAN only, and L1 + GAN (Eq. 4). Predict and then
            measure the interior contrast (how sharp/decisive the fill is, $0$=gray $\approx$ blur, $1$=committed)
            and the faithfulness (does the output match the input's shape). Which term buys sharpness, which buys
            faithfulness, and why is the combination best?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Train L1 only (drop the GAN term). On the ambiguous fill, the L1 minimizer is the median over the plausible textures. — _When several outputs are valid, a distance loss outputs their central value &mdash; the average of competing sharp textures is a flat gray (the paper's blur, §1)._
- Train GAN only (drop the L1 term). Output gets sharp but can drift in content / show artifacts. — _The discriminator rewards realism, not faithfulness to this exact $x$; nothing anchors the output to the ground truth._
- Train L1 + GAN (Eq. 4, $\lambda=100$). Measure interior contrast and shape match. — _L1 supplies the low-frequency content (right shape/colors), the GAN supplies the high-frequency sharpness &mdash; the division of labor the paper describes._
- Conclude: GAN&rarr;sharpness, L1&rarr;faithfulness, sum&rarr;both; matches Table 1 where L1+cGAN scores highest. — _This is the paper's central ablation, reproduced qualitatively on toy data._

**Answer:** L1 only blurs, L1+GAN commits. In our toy run the L1-only generator hedges the
                 ambiguous fill toward a washed-out gray &mdash; its interior spatial std collapses to about
                 0.000 (a flat constant, versus the target's 1.004) &mdash; because the L1 optimum on a
                 multimodal target is a central (averaged) value. The L1+PatchGAN generator instead commits to
                 one sharp, real-looking stripe phase &mdash; interior std about 1.004, matching the target
                 &mdash; while the heavily-weighted L1 term ($\lambda=100$) keeps the overall shape correct. The
                 L1-only loss also plateaus at an irreducible floor (~0.317) it cannot beat by averaging,
                 whereas adding the GAN lets $G$ dip below it (~0.198) by picking a single mode. This is
                 exactly the paper's Table 1 story (L1+cGAN best): the GAN term buys sharpness, the L1 term buys
                 faithfulness, and you want both.

</details>

**Problem 2.** Your worked example had a true pixel $y=0.7$ and an output $G(x)=-0.3$, so the L1 cost was
            $|0.7-(-0.3)|=1.0$ and $\lambda$ times it was $100$. Now suppose for some input the correct pixel is
            equally likely $+1$ or $-1$. What constant output minimizes the expected L1 cost, what is that cost, and
            what does this say about why L1 alone blurs?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The expected L1 for a constant $c$ is $\tfrac12|c-1| + \tfrac12|c+1|$. — _Average the two equally-likely targets._
- This is minimized for any $c\in[-1,1]$ (the median is not unique here); the symmetric pick is $c=0$, cost $\tfrac12(1)+\tfrac12(1)=1$. — _L1's minimizer is the conditional median; with two symmetric modes the middle (gray) is optimal._
- So L1 has no reason to commit to $+1$ or $-1$ &mdash; it outputs the gray middle, which renders as blur. — _The blur is the optimum of the loss, not a training failure &mdash; the GAN term is what penalizes it._

**Answer:** The expected L1 cost is $\tfrac12|c-1|+\tfrac12|c+1|$, minimized by the median &mdash; here
                 any $c\in[-1,1]$, with the symmetric choice $c=0$ giving cost $1$. So when the target is ambiguous,
                 L1 is perfectly happy outputting the gray middle instead of committing to $+1$ or $-1$; that
                 central value is the blur. The GAN term breaks the tie, because a gray patch is easily flagged as
                 fake, forcing $G$ to pick one sharp mode.

</details>

**Problem 3.** Why does pix2pix make the discriminator judge the pair $(x, \text{output})$ rather than just the
            output image, and what failure appears if you feed $D$ only the output?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the goal is translation: the output must correspond to this input $x$, not just look real. — _A realistic photo that ignores the input label map is a wrong translation._
- Feeding $D$ the pair lets it check "is this a real $(x,y)$ correspondence?" &mdash; conditioning, as in paper-cgan. — _Only the pair carries the information of whether output matches input._
- If $D$ sees only the output, it can be fooled by any realistic image; $G$ is free to ignore $x$ (mode collapse onto generic realistic outputs). — _Without the pair, there is no pressure to respect the input &mdash; the L1 term alone must carry all faithfulness._

**Answer:** Because translation requires the output to match the specific input $x$, not merely look
                 real. Feeding $D$ the concatenated pair $(x,\text{output})$ makes it a "is this a real correspondence
                 of $x$?" judge (the conditional GAN of paper-cgan). If $D$ sees only the output, any realistic image
                 passes and $G$ can ignore $x$ &mdash; you lose the conditioning and lean entirely on L1 for
                 faithfulness, which then blurs. The pair is what ties realism to the right input.

</details>